# Copa do Mundo 2026 - Análise Cartola Copa

Análise de desempenho das seleções e jogadores com dados do Cartola Copa.

In [5]:
import glob
import pandas as pd

files = sorted(glob.glob("data/pontuados-rodada-*.csv"))
dfs = []
for f in files:
    df = pd.read_csv(f)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Total registros: {len(df_all)}")
print(f"Rodadas: {files}")
print(f"Seleções: {df_all['clube_nome'].nunique()}")
print(f"Jogadores: {df_all['atleta_id'].nunique()}")


ModuleNotFoundError: No module named 'pandas'

## Mapa de posições

In [ ]:
POSICOES = {
    1: "Goleiro",
    2: "Lateral",
    3: "Zagueiro",
    4: "Meia",
    5: "Atacante",
    6: "Técnico",
}
df_all["posicao"] = df_all["posicao_id"].map(POSICOES)


## Ranking de Seleções por Pontuação Total

Soma dos pontos de todos os jogadores que entraram em campo, por seleção.

In [ ]:
em_campo = df_all[df_all["entrou_em_campo"] == True]

ranking_selecoes = (
    em_campo.groupby("clube_nome")
    .agg(
        pontuacao_total=("pontuacao", "sum"),
        media_pontuacao=("pontuacao", "mean"),
        jogadores_usados=("atleta_id", "nunique"),
        gols=("G", "sum"),
        assistencias=("A", "sum"),
        jogos=("atleta_id", "count"),
    )
    .sort_values("pontuacao_total", ascending=False)
    .round(2)
)

print("=" * 60)
print("RANKING SELEÇÕES - PONTUAÇÃO TOTAL")
print("=" * 60)
ranking_selecoes.head(20)


## Ranking de Seleções por Média de Pontuação

Média por jogador — mede qualidade técnica independente de quantos jogaram.

In [ ]:
ranking_media = ranking_selecoes.sort_values("media_pontuacao", ascending=False)

print("=" * 60)
print("RANKING SELEÇÕES - MÉDIA POR JOGADOR")
print("=" * 60)
ranking_media.head(20)


## Top 30 Jogadores por Pontuação Total

In [ ]:
top_jogadores = (
    em_campo.groupby(["atleta_id", "apelido", "clube_nome", "posicao"])
    .agg(
        pontuacao_total=("pontuacao", "sum"),
        media_pontuacao=("pontuacao", "mean"),
        rodadas=("pontuacao", "count"),
        gols=("G", "sum"),
        assistencias=("A", "sum"),
    )
    .sort_values("pontuacao_total", ascending=False)
    .round(2)
    .head(30)
)

top_jogadores


## Top Jogadores por Posição

In [ ]:
for pos in ["Goleiro", "Zagueiro", "Lateral", "Meia", "Atacante", "Técnico"]:
    top = (
        em_campo[em_campo["posicao"] == pos]
        .groupby(["apelido", "clube_nome"])
        .agg(
            pontuacao_total=("pontuacao", "sum"),
            gols=("G", "sum"),
            assistencias=("A", "sum"),
            SG=("SG", "sum"),
        )
        .sort_values("pontuacao_total", ascending=False)
        .round(2)
        .head(10)
    )
    print(f"\n{'=' * 50}")
    print(f"TOP 10 - {pos.upper()}")
    print(f"{'=' * 50}")
    display(top)


## Scouts Agregados por Seleção

Perfil técnico: gols, assistências, desarmes, finalizações, faltas.

In [ ]:
scout_cols = ["G", "A", "FT", "FD", "FF", "FS", "FC", "DE", "DS", "SG", "CA", "CV", "GC", "GS"]
available = [c for c in scout_cols if c in em_campo.columns]

scout_labels = {
    "G": "Gols", "A": "Assistências", "FT": "Finalizações Certas",
    "FD": "Finalizações Defendidas", "FF": "Finalizações Fora",
    "FS": "Faltas Sofridas", "FC": "Faltas Cometidas",
    "DE": "Desarmes", "DS": "Dribles", "SG": "Saldo de Gols (DEF)",
    "CA": "Cartões Amarelos", "CV": "Cartões Vermelhos",
    "GC": "Gols Contra", "GS": "Gols Sofridos (GOL)",
}

perfil = (
    em_campo.groupby("clube_nome")[available]
    .sum()
    .fillna(0)
    .astype(int)
    .sort_values("G", ascending=False)
)
perfil.columns = [scout_labels.get(c, c) for c in perfil.columns]

print("PERFIL TÉCNICO POR SELEÇÃO")
perfil.head(20)


## Evolução por Rodada

In [ ]:
rodada_col = [c for c in df_all.columns if 'rodada' in c.lower()]
if not rodada_col:
    df_all["rodada"] = df_all.groupby("atleta_id").cumcount() + 1
    rodada_col = ["rodada"]

# Infer rodada from file order
dfs_labeled = []
for i, f in enumerate(files, 1):
    df_temp = pd.read_csv(f)
    df_temp["rodada"] = i
    dfs_labeled.append(df_temp)

df_rodadas = pd.concat(dfs_labeled, ignore_index=True)
df_rodadas["posicao"] = df_rodadas["posicao_id"].map(POSICOES)
em_campo_r = df_rodadas[df_rodadas["entrou_em_campo"] == True]

evolucao = (
    em_campo_r.groupby(["rodada", "clube_nome"])
    .agg(pontuacao_total=("pontuacao", "sum"))
    .reset_index()
    .pivot(index="clube_nome", columns="rodada", values="pontuacao_total")
    .fillna(0)
)
evolucao["total"] = evolucao.sum(axis=1)
evolucao = evolucao.sort_values("total", ascending=False)

print("PONTUAÇÃO POR RODADA")
evolucao.head(20)
